# Пара 2. Бунин в Google Colab: дуга тональности, локации, сравнение

Продолжение пары 1. Цель: две «картинки смысла» — эмоциональная дуга и сравнение пространства — и проверка цитатой.


## 0. Загрузка текстов
Если память Colab сбросилась — запустите эту ячейку снова.


In [ ]:
import re
import math
import requests
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

BASE = "https://pavelaleks.github.io/samarkand_lectures/demo-corpora/bunin-temnye-allei/txt/"
STORIES = {
    "Кавказ": "kavkaz.txt",
    "Холодная осень": "holodnaya-osen.txt",
    "Чистый понедельник": "chistyj-ponedelnik.txt",
}

texts = {}
for title, fname in STORIES.items():
    r = requests.get(BASE + fname, timeout=60)
    r.raise_for_status()
    r.encoding = "utf-8"
    texts[title] = r.text
    print(title, "OK", len(texts[title].split()), "слов~")


## 1. Учебный словарь тональности
Простой sentiment (+1/−1). Это гипотеза о лексическом тоне, а не «душа автора». Особенно хрупко на иронии и церковной лексике.


In [ ]:
POS = ["счаст", "радост", "нежн", "мил", "светл", "люб", "прекрас", "весел", "тепл", "блажен"]
NEG = ["страш", "ужас", "смерт", "холод", "гор", "тоск", "мук", "зло", "черн", "пуст", "разлук", "плак", "рыда"]

def tokenize(text):
    return re.findall(r"[А-Яа-яЁёA-Za-z]+", text.lower())

def chunk_text(text, n_chunks=8):
    parts = re.split(r"(?<=[.!?…])\s+", text.strip())
    parts = [p for p in parts if p.strip()]
    if len(parts) < n_chunks:
        words = text.split()
        size = max(1, len(words) // n_chunks)
        return [" ".join(words[i*size:(i+1)*size]) for i in range(n_chunks)]
    size = max(1, len(parts) // n_chunks)
    chunks = []
    for i in range(n_chunks):
        chunk = parts[i*size:(i+1)*size] if i < n_chunks - 1 else parts[i*size:]
        chunks.append(" ".join(chunk))
    return chunks

def score_chunk(text):
    toks = tokenize(text)
    if not toks:
        return 0.0
    s = 0
    for t in toks:
        if any(t.startswith(p) for p in POS):
            s += 1
        if any(t.startswith(n) for n in NEG):
            s -= 1
    return s / math.sqrt(len(toks))

story = "Холодная осень"
chunks = chunk_text(texts[story], n_chunks=8)
scores = [score_chunk(c) for c in chunks]

plt.figure(figsize=(9, 4))
plt.plot(range(1, 9), scores, marker="o")
plt.axhline(0, color="gray", lw=1)
plt.xlabel("часть текста (1→8)")
plt.ylabel("индекс тона (учебный)")
plt.title(f"Эмоциональная дуга: {story}")
plt.tight_layout()
plt.show()

for i, (sc, ch) in enumerate(zip(scores, chunks), 1):
    preview = re.sub(r"\s+", " ", ch)[:110]
    print(f"{i}: score={sc:+.2f} | {preview}...")


### Мини-разбор (8–10 минут)
Найдите пик и спад. Это эмоция героя, тон повествователя или ложное срабатывание словаря?


In [ ]:
arc_note = '''
Пик/спад объясняется так: ...
Риск словаря: ...
'''
print(arc_note)


## 2. Учебные локации (без тяжёлого NER)
Кураторский список мест + подсчёт. Дома можно сравнить с Natasha.


In [ ]:
PLACE_CANDIDATES = [
    "москва", "москве", "москвы", "петербург", "петербурге",
    "кавказ", "кавказе", "крым", "ялта", "ялте",
    "ордынка", "ордынке", "замоскворечье",
    "париж", "париже",
    "россия", "россии", "деревн", "станция", "вокзал",
    "монастыр", "церкв", "храм", "кладбищ", "ресторан", "театр",
]

def place_counts(text):
    toks = tokenize(text)
    bag = Counter()
    for t in toks:
        for p in PLACE_CANDIDATES:
            if t.startswith(p):
                canon = p
                for c in ["москв", "кавказ", "ялт", "ордын", "париж", "росси",
                          "монастыр", "церкв", "храм", "кладбищ", "деревн", "театр", "ресторан"]:
                    if p.startswith(c):
                        canon = c
                        break
                bag[canon] += 1
                break
    return bag

cmp_rows = []
for title, text in texts.items():
    bag = place_counts(text)
    N = max(1, len(tokenize(text)))
    for place, cnt in bag.items():
        cmp_rows.append({
            "рассказ": title,
            "место": place,
            "упоминаний": cnt,
            "на_1000": round(1000 * cnt / N, 2),
        })

place_df = pd.DataFrame(cmp_rows)
place_df.sort_values(["рассказ", "на_1000"], ascending=[True, False])


In [ ]:
pivot = place_df.pivot_table(index="место", columns="рассказ", values="на_1000", fill_value=0)
pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

plt.figure(figsize=(8, 5))
plt.imshow(pivot.values, aspect="auto")
plt.yticks(range(len(pivot.index)), pivot.index)
plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=15)
plt.colorbar(label="на 1000 токенов")
plt.title("Учебная карта мест (по списку-кандидатов)")
plt.tight_layout()
plt.show()
pivot.head(12)


## 3. Сравнение «Кавказ» и «Чистый понедельник»


In [ ]:
focus = ["москв", "кавказ", "ялт", "ордын", "монастыр", "церкв", "ресторан", "театр", "париж"]
sub = place_df[place_df["место"].isin(focus) & place_df["рассказ"].isin(["Кавказ", "Чистый понедельник"])]
piv = sub.pivot(index="место", columns="рассказ", values="на_1000").fillna(0)

ax = piv.plot(kind="bar", figsize=(9, 4), rot=20)
ax.set_ylabel("на 1000 токенов")
ax.set_title("Пространственные маркеры: Кавказ и Чистый понедельник")
plt.tight_layout()
plt.show()


## 4. Синтез (15 минут)
1. Как дуга «Холодной осени» стыкуется с финалом?
2. Чем пространство «Кавказа» отличается от «Чистого понедельника»?
3. Какой график был эвристикой, а какой почти сорвался из-за метода?


In [ ]:
synthesis = '''
1) ...
2) ...
3) ...
'''
print(synthesis)


## 5. Мост дальше
Сохраните блокнот на Диск. Красивый график не равен выводу: нужны цитата и ограничение метода.
Эти навыки пригодятся в заданиях ХЕМИС.


In [ ]:
plt.figure(figsize=(6, 3))
plt.plot([1, 2, 3], [0.1, -0.2, 0.05], marker="o")
plt.title("пример сохранения графика")
plt.savefig("primer_grafik.png", dpi=150, bbox_inches="tight")
print("Файл primer_grafik.png появился слева в файлах Colab")
